<!-- 학습 보강 셀 -->

# 03. Node 학습 흐름

이 노트북은 `Document`와 `Node`의 차이를 이해하는 데 초점을 둡니다.
Document가 원본 문서라면, Node는 검색과 임베딩에 쓰기 위해 잘라낸 조각입니다.

### 1. Node 객체 직접 생성

<!-- 학습 보강 셀 -->

## 왜 Document를 바로 검색하지 않고 Node로 나눌까?

긴 문서 전체를 하나의 벡터로 만들면 질문과 관련된 작은 부분을 찾기 어렵습니다.
Node로 나누면 검색 단위가 작아져서 특정 문장이나 문단을 더 정확히 찾을 수 있습니다.
RAG 품질은 Node를 얼마나 적절한 크기로 나누는지에 크게 영향을 받습니다.

In [ ]:
from llama_index.core import Document
from llama_index.core.schema import TextNode

# Document는 원본 문서이고, Node는 검색/임베딩에 사용하기 위해 나눈 작은 조각입니다.
document = Document(
    text="""인공지능은 우리의 미래를 변화시킬 것입니다.
이러한 변화에 우리는 준비되어 있어야 합니다.""",
    id_='ai_future_doc',
)

# 문서를 2개의 노드로 직접 분할합니다.
# - TextNode의 고유 ID는 id_로 지정합니다.
# - 원본 Document와의 연결 정보는 metadata에 남겨 추적할 수 있게 합니다.
# - 기존처럼 고정 글자 수로 자르면 문장이 중간에서 끊길 수 있어 문장 단위로 나눕니다.
sentences = [line.strip() for line in document.text.splitlines() if line.strip()]
node1 = TextNode(
    text=sentences[0],
    id_='ai_future_node_1',
    metadata={'source_document_id': document.id_},
)
node2 = TextNode(
    text=sentences[1],
    id_='ai_future_node_2',
    metadata={'source_document_id': document.id_},
)

print(node1)
print(node2)

### 2. 문서를 불러와서 Document와 Node 객체 생성


In [ ]:
# Word(.docx) 파일을 읽으려면 docx2txt가 필요합니다.
# !pip install docx2txt

<!-- 학습 보강 셀 -->

## 파일 형식별 추가 패키지

PDF, Word, Excel, 웹 페이지처럼 파일 형식이 달라지면 내부 파서도 달라집니다.
`SimpleDirectoryReader`는 공통 인터페이스를 제공하지만, 실제 파일을 읽기 위해서는 형식별 의존성이 필요할 수 있습니다.
실행 오류가 나면 먼저 해당 파일 형식의 reader 패키지가 설치되어 있는지 확인하세요.

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

In [ ]:
# NewData 폴더에서 docx 파일만 읽습니다.
# required_exts를 지정하면 다른 확장자는 무시되므로 예제가 안정적으로 실행됩니다.
documents = SimpleDirectoryReader(
    input_dir='../NewData',
    required_exts=['.docx'],
).load_data()
print('읽어온 문서 수:', len(documents))

In [ ]:
# 읽어온 Word 문서의 본문 일부를 확인합니다.
print(documents[0].text[:1000])

In [ ]:
# SentenceSplitter는 문장 경계를 최대한 보존하면서 문서를 Node로 분할합니다.
# chunk_size는 한 Node의 최대 토큰 수, chunk_overlap은 앞뒤 Node 사이에 겹쳐 둘 토큰 수입니다.
parser = SentenceSplitter(
    chunk_size=200,
    chunk_overlap=20,
)

nodes = parser.get_nodes_from_documents(documents)
print('생성된 Node 수:', len(nodes))

<!-- 학습 보강 셀 -->

## chunk_size와 chunk_overlap 감각 잡기

`chunk_size`가 너무 작으면 문맥이 잘려 답변 품질이 떨어질 수 있고, 너무 크면 검색이 둔해질 수 있습니다.
`chunk_overlap`은 앞뒤 조각 사이에 문맥을 조금 겹쳐 두는 장치입니다.
일반적으로 문단 단위 의미가 유지되는지 출력된 Node를 직접 읽어 보며 조정합니다.

In [ ]:
# Node 확인
# - enumerate(..., start=1)을 사용하면 출력 번호가 사람이 읽기 편한 1부터 시작합니다.
for i, node in enumerate(nodes, start=1):
    print(f'=== Node {i} ===')
    print(node.text)
    print('-' * 80)

In [ ]:
# 첫 번째 Node의 메타데이터 확인
# - 이전 셀의 반복문 변수 node에 의존하지 않도록 nodes[0]을 직접 사용합니다.
nodes[0].metadata